# 機械学習を試してみよう
---
## 目的
いくつかの教師あり学習を活用して，実環境問題を解く．

In [23]:
%matplotlib inline
# pandasをpdとしてimportする．
import pandas as pd
# 機械学習ライブラリsklearnからdatasetsをimportする．
from sklearn import datasets

import matplotlib.pyplot as plt
import numpy as np
import warnings

warnings.simplefilter('ignore')

### 教師あり学習
学習データに正解（教師データ）を与え，その規則性を学習する手法．<br>
つまり，`{ 特徴量ベクト, 教師ラベル }`のペアの集合体（データセット）を元に分類や回帰のルールを自動で学習する手法である．<br>

学習アルゴリズムは様々なものがある．
- ニューラルネットワーク (NN: Neural Network)
- ナイーブベイズ (NB: Naive Bayes)
- 決定木 (DT: Decision Tree)
- ランダムフォレスト (RF: Random Forest)
- サポートベクターマシン (SVM: Support Vector Machine)

## パーセプトロンを試す
今回は，ニューラルネットワークの元となった単純パーセプトロン（Perceptron）を試す．<br>
（第２回講義で説明予定．）

1. データセットの準備：<br>
【データセット名】Iris Dataset<br>
「あやめ」の花での形状データを格納したデータセット．<br>
本データセットは以下の 4 つの特徴で表される．<br>
　a. がく片の長さ（cm）：Sepal Length<br>
　b. がく片の幅（cm）：Sepal Width<br>
　c. 花弁の長さ（cm）：Petal Length<br>
　d. 花弁の幅（cm）：Petal Width<br>

In [24]:
# irisデータをロード
iris = datasets.load_iris()
# irisはクラスなので，iris.dataをDataFrameに変換する．
X = pd.DataFrame(iris.data, columns=["sepalLen", "sepalWid", "petalLen", "petalWid"])
# 分類対象はyとしておく
y = iris.target_names[iris.target]
# 上5行を表示する．
X.head()

,sepalLen,sepalWid,petalLen,petalWid
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


2. 機械学習モデルを定義：
今回は手軽に機械学習アルゴリズムを試すために，`scikit-learn (sklearn)`ライブラリを用いる．<br>
scikit-learnは，幅広いアルゴリズムをサポートしているだけでなく，ほとんどのアルゴリズムを同じ形式で使えるため比較検討に非常に便利なライブラリである．

In [25]:
# sklearnのimport
from sklearn.linear_model import Perceptron

# モデル定義：パーセプトロンインスタンス生成
ppn = Perceptron(max_iter=40, eta0=0.1, random_state=0, shuffle=True)
# トレーニングデータをモデルに適合
ppn.fit(X, y)
# トレーニングしたモデルでXを予測してみる．
y_pred = ppn.predict(X)

# 推定精度の表示
print('===推定精度======================')
print('誤分類:%d' % (y != y_pred).sum())
print('正解率: %.2f' % ((y == y_pred).sum()/len(y)))

===推定精度======================
誤分類:78
正解率: 0.48


```
Perceptron(max_iter=40, eta0=0.1, random_state=0, shuffle=True)
```
学習を行うモデルをインスタンス化している．<br>
引数は学習の効率に影響するパラメータ，即ち，人為的に調整が可能な計数である．
パラメータの詳細は次回以降で触れていく．

```
ppn.fit(X, y)
```
続いて，学習フェーズである．<br>
入力特徴量`X`に対して，教師ラベル`y`が与えられるとき，上記で学習を行う．

```
y_pred = ppn.predict(X)
```
学習が完了したモデル`ppn`を用いて，もう一度入力特徴量`X`を与えて予測を行ってみる．<br>
**この時，教師ラベル`y`は与えない．**<br>
つまり，一度学習を行えば，正解が未知の入力が与えられたときに予測が実践できるということを確認している．

```
(y == y_pred).sum()/len(y)
```
分類精度の評価は教師ラベル`y`とどれだけ一致したかとなるため，正解率は上記で算出できる．<br>
`(y == y_pred)`は教師ラベルと予測結果の一致を確認しており，boolean配列となる．<br>
これを`.sum()`することで`True`を`1`と見立てた加算が行われる．
それを`len(y)`，即ち教師ラベルの数で割ると正解率となる．

## 精度向上に向けて
正解率が`48%`と，予想以上に低かったことを受け，少し調整を行ってみよう．まず，前処理であるスケーリングを実施してみる．

In [26]:
# 正規化，標準化
from sklearn.preprocessing import MinMaxScaler,StandardScaler

# 正規化：min-max normalization
mms = MinMaxScaler()
X2 = mms.fit_transform(X)

# モデル定義：パーセプトロンインスタンス生成
ppn = Perceptron(max_iter=40, eta0=0.1, random_state=0, shuffle=True)
# トレーニングデータをモデルに適合
ppn.fit(X2, y)
# トレーニングしたモデルでXを予測してみる．
y_pred = ppn.predict(X2)
print('===Min-Max Normalized Perceptron======================')
print('誤分類:%d' % (y != y_pred).sum()) # 推定精度の表示
print('正解率: %.2f' % ((y == y_pred).sum()/len(y)))


# 標準化：z-score normalization
ss = StandardScaler()
X3 = ss.fit_transform(X)

# モデル定義：パーセプトロンインスタンス生成
ppn = Perceptron(max_iter=40, eta0=0.1, random_state=0, shuffle=True)
# トレーニングデータをモデルに適合
ppn.fit(X3, y)
# トレーニングしたモデルでXを予測してみる．
y_pred = ppn.predict(X3)
print('===z-score Normalized Perceptron======================')
print('誤分類:%d' % (y != y_pred).sum()) # 推定精度の表示
print('正解率: %.2f' % ((y == y_pred).sum()/len(y)))

===Min-Max Normalized Perceptron======================
誤分類:14
正解率: 0.91
===z-score Normalized Perceptron======================
誤分類:18
正解率: 0.88


推定精度が若干向上した様子が確認できる．<br>

## 様々なアルゴリズムを試す
せっかくなので，色々なアルゴリズムを試してみる．

In [27]:
from sklearn.neural_network import MLPClassifier
# Multi layer perceptron（多層パーセプトロン）
nn = MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 2), random_state=1)
nn.fit(X3, y)
y_pred = nn.predict(X3)
print('Neural Network \t accuracy:%.2f' % ((y == y_pred).sum()/len(y)) )

from sklearn.naive_bayes import GaussianNB
# GaussianNBはガウス分布ベースのNaive Bayes
nb = GaussianNB()
nb.fit(X3, y)
y_pred = nb.predict(X3)
print('Naive Bayes \t accuracy:%.2f' % ((y == y_pred).sum()/len(y)) )

from sklearn.tree import DecisionTreeClassifier
# Decision Tree　決定木
dt = DecisionTreeClassifier()
dt.fit(X3, y)
y_pred = dt.predict(X3)
print('Decision Tree \t accuracy:%.2f' % ((y == y_pred).sum()/len(y)) )

from sklearn.ensemble import RandomForestClassifier
# Random Forest　ランダムフォレスト
rf = RandomForestClassifier(n_estimators=100, max_depth=2, random_state=0)
rf.fit(X3, y)
y_pred = rf.predict(X3)
print('Random Forest \t accuracy:%.2f' % ((y == y_pred).sum()/len(y)) )

from sklearn.svm import SVC
# SVM : Support Vector Machine の Classification（分類）なのでSVC
svm = SVC(gamma='auto')
svm.fit(X3, y)
y_pred = svm.predict(X3)
print('SVM       \t accuracy:%.2f' % ((y == y_pred).sum()/len(y)) )

Neural Network 	 accuracy:0.99
Naive Bayes 	 accuracy:0.96
Decision Tree 	 accuracy:1.00
Random Forest 	 accuracy:0.97
SVM       	 accuracy:0.97


## 学習データと検証データ
上記の結果では推定精度100%を達成しているが，これは正しい評価なのだろうか？

現状では`{ X, y }`のペアを学習して，`{ X, ? }`を予測しており，既に学習したことのあるデータで予測を行っている．<br>
即ち，テストの問題を事前にカンニングして高い点数を取っているのと同じなのである．<br>

そこで，モデルの検証を行う方法として，データセットを学習データと検証データに分割するという方法がある．<br>
この方法では，データセットの9割を学習用に，1割を検証用に事前に分割し，検証用データの推定精度を用いてモデルの検証を行う．

ただし，分割の際には先頭15個：残り135個のような分割を行うのはよろしくない．`y`を見てみるとわかるが，これだとsetosaが15個と残り135個のデータに分割されてしまい，クラスが非常に偏ってしまう．一般的にこのようなケースでは**ランダムサンプリング**することで対応を行う．もしくは，クラスの偏りを考慮したランダムサンプリングを行う．

In [28]:
import random

# 単純なランダムサンプリング
size = len(X3)
# random.sample(選択の範囲, 選択する数)→今回は0～size-1をランダム順に全て取得している．
indexes = random.sample(range(size), size)
# ランダム順なので，先頭1割を検証用indexに，残りを学習用indexにする．
test = indexes[:(int)(size/10)]
train = indexes[(int)(size/10):]

# {X3,y}は現在numpy型配列なので以下のようにして分割を記述する．
X_train, y_train = X3[train], y[train]
X_test, y_test = X3[test], y[test]

print(X_test, y_test)

[[ 1.76501198 -0.36217625  1.44480739  0.79067065]
 [-1.74885626 -0.13197948 -1.39706395 -1.3154443 ]
 [ 0.67450115  0.32841405  0.87643312  1.44883158]
 [ 0.55333328 -1.74335684  0.36489628  0.13250973]
 [-0.7795133   1.01900435 -1.2833891  -1.3154443 ]
 [-0.65834543  1.47939788 -1.2833891  -1.3154443 ]
 [-1.02184904 -2.43394714 -0.14664056 -0.26238682]
 [ 1.03800476 -0.13197948  0.70592084  0.65903847]
 [-0.7795133  -0.82256978  0.08070915  0.26414192]
 [-0.41600969  1.01900435 -1.39706395 -1.3154443 ]
 [-0.29484182 -1.28296331  0.08070915 -0.13075464]
 [ 0.55333328  0.55861082  1.27429511  1.71209594]
 [ 2.24968346 -0.13197948  1.33113254  1.44883158]
 [ 0.55333328 -0.36217625  1.0469454   0.79067065]
 [ 1.52267624 -0.13197948  1.21745768  1.18556721]] ['virginica' 'setosa' 'virginica' 'versicolor' 'setosa' 'setosa'
 'versicolor' 'versicolor' 'versicolor' 'setosa' 'versicolor' 'virginica'
 'virginica' 'virginica' 'virginica']


In [29]:
# 続いて，学習フェーズ，予測フェーズに分割してモデルを評価してみよう．
iter=100
ppn = Perceptron(max_iter=iter, eta0=0.1, random_state=0, shuffle=True)
ppn.fit(X_train, y_train)
y_pred = ppn.predict(X_test)
accuracies = list()
accuracies.append((y_test == y_pred).sum()/len(y_test))
print('Perceptron \t accuracy:%.2f' % ((y_test == y_pred).sum()/len(y_test)) )

Perceptron 	 accuracy:0.93


Scikit-learnの特徴である「ほとんどのアルゴリズムを同じ形式で使える」を活用して，プログラムを記述する．<br>
学習モデルを辞書型｛名前:中身｝で保持する．これは，Key-ValueペアとかHashMap的なものである．

In [30]:
ESTIMATORS = {"Perceptron": Perceptron(max_iter=iter, eta0=0.1, random_state=0, shuffle=True),
              "Neural Network": MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 2), random_state=1),
              "Naive Bayes": GaussianNB(),
              "Decision Tree": DecisionTreeClassifier(),
              "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=2, random_state=0),
              "SVM       ": SVC(gamma='auto')}

# ESTIMATORS全てに対して，以下を実施する．
for name, clf in ESTIMATORS.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print('%s \t accuracy:%.2f' % (name, (y_test == y_pred).sum()/len(y_test)) )

Perceptron 	 accuracy:0.93
Neural Network 	 accuracy:0.93
Naive Bayes 	 accuracy:0.93
Decision Tree 	 accuracy:0.93
Random Forest 	 accuracy:0.93
SVM        	 accuracy:0.93


## 交差検証
上記によって，カンニングは防止することができたが，150個のデータのうち，15個でしか評価していない．すると以下のような課題が残る．
- ランダムの15個が偏ってたまたま高い精度になったのでは？
- 15個だけって評価材料が少なくない？

だからといって，検証用データの比率を増加させると，「機械学習は学習データが多いほど高い精度となる」に反するため，本来の能力を計測できているとはいいがたい．

そこで，データセットを無駄なく使ってモデルを評価する手法として，交差検証（Cross Validation）がある．ここではthe most populerな10-fold CVを例に，交差検証の方法を説明する．

1. データセットをランダムサンプリングにより10分割する（サブセットA～Jとする）．
1. Aを検証用に，残りを学習用にして，Aに対する予測結果を出力する（pred<sub>A</sub>とする）．
1. Bを検証用に，同上する．
1. …と10セット全てに対して実施する（pred<sub>A</sub>～pred<sub>J</sub>）．
1. これらを結合したpred<sub>A2J</sub>と，対応する教師ラベルy<sub>A2J</sub>を用いて全体の精度評価とする．

In [31]:
# 交差検証は自分で実装してもよいが便利なクラスが既に用意されている．
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True)

# skfは各クラスの比率を元のデータセットと同じにしながらK-foldするクラスである．
# 例えばclf=RandomForest()を評価してみよう．
clf = RandomForestClassifier(n_estimators=100, max_depth=2, random_state=0)
result = pd.DataFrame() # ここに結果を残そう．
for train, test in skf.split(X3, y):
    clf.fit(X3[train], y[train])
    y_pred = clf.predict(X3[test])
    result = pd.concat([result,pd.DataFrame({"y": y[test], "pred": y_pred})], ignore_index=True)
print('Random Forest \t accuracy:%.2f' % ((result["y"] == result["pred"]).sum()/len(result)) )

Random Forest 	 accuracy:0.95


In [32]:
# 交差検証は自分で実装してもよいが便利なクラスが既に用意されている．
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True)

# ESTIMATORS全てに対して，以下を実施する．
for name, clf in ESTIMATORS.items():
    result = pd.DataFrame() # ここに結果を残そう．
    # skfは各クラスの比率を元のデータセットと同じにしながらK-foldするクラスである．
    for train, test in skf.split(X3, y):
        clf.fit(X3[train], y[train])
        y_pred = clf.predict(X3[test])
        result = pd.concat([result,pd.DataFrame({"y": y[test], "pred": y_pred})], ignore_index=True)
    print('%s \t accuracy:%.2f' % (name, (result["y"] == result["pred"]).sum()/len(result)) )

Perceptron 	 accuracy:0.83
Neural Network 	 accuracy:0.95
Naive Bayes 	 accuracy:0.95
Decision Tree 	 accuracy:0.95
Random Forest 	 accuracy:0.95
SVM        	 accuracy:0.96
